### Obtaining the Movie Review Dataset 

We will be working with a large dataset of movie reviews from IMDb that has been collected by Andrew Maas and others. The movie review dataset consists of 50,000 polar movie reviews that are labeled as either positive or negative. Here, positive means the movie was rated with more than six stars on IMDb, and negative means that a movie was rated with fewer than five stars on IMDb. We will start by downloading the dataset, preprocessing it into a useable format for machine learning tools, and extracting meaningful information from a subset of these movie reviews to build a machine learning model that can predict whether a certain reviewer liked or disliked a movie.

The compressed dataset can be downloaded from https://ai.stanford.edu/~amaas/data/sentiment/ as a gzip-compressed tarball archive. You can unpack it using the following code:

In [1]:
import tarfile
from pathlib import Path

if not Path('aclImdb').is_dir():
    with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
        tar.extractall()

### Preprocessing the Movie Dataset into a More Convenient Format

Having successfully extracted the dataset, we will now assemble the inidividual text documents from the decompressed download archive into a single CSV file. In the following code section, we will be reading the movie reviews into a pandas DataFrame object, which can take up some time on a standard desktop computer:

In [2]:
import pandas as pd
import os
import sys
basepath = 'aclImdb'

labels = {'pos': 1, 'neg': 0}
rows = []

for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file),
                      'r', encoding='utf-8') as infile:
                  txt = infile.read()
            rows.append([txt, labels[l]])
            
df = pd.DataFrame(rows, columns=['review', 'sentiment'])

In the preceding code, we used nested for loops to iterate over the train and test subdirectories in the main aclImdb directory and read the individual text files from the pos and neg subdirectories that we appended to a list to eventually make a DataFrame object, alongside the integer class labels (1 = positive, 0 = negative).

Since the class labels in the assembled dataset are sorted, we will now shuffle the DataFrame using the permutation function from the np.random submodule - this will be useful for splitting the dataset into training and test datasets in later sections, when we will stream the data from our local drive directly. For our own convenience, we will also store the assembled and shuffled movie review dataset as a CSV file:

In [3]:
import numpy as np
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')

Since we are going to use this dataset later in this chapter, let's quickly confirm that we have successfully saved the data in the right format by reading in the CSV and printing an excerpt of the first three examples:

In [4]:
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


### Introducing the Bag-of-Words Model

To construct a bag-of-words model based on the word counts in the respective documents, we can use the CountVectorizer class implemented in scikit-learn:

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer()
docs = np.array(['The sun is shining',
                 'The weather is sweet',
                 'The sun is shining, the weather is sweet, and one and one is two' ])
bag = count.fit_transform(docs)

By calling the fit_transform method on CountVectorizer, we constructed the vocabulary of the bag-of-words model and transformed the three sentences into sparse feature vectors.

Now let's print the contents of the vocabulary to get a better understanding of the underlying concepts:

In [6]:
print(count.vocabulary_)

{'the': 6, 'sun': 4, 'is': 1, 'shining': 3, 'weather': 8, 'sweet': 5, 'and': 0, 'one': 2, 'two': 7}


In [7]:
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]


As we can see from the preceding commands, the vocabulary is stored in a Python dictionary that maps the unique words to integer indices. Each index position in the feature vectors correspond to the integer values that are stored as dictionary items in the CountVectorizer vocabulary. For example, the first feature at index position 0 resembles the count of the word 'and', which only occurs in the last document, and the word 'is' at index position 1, occurs in all three sentences.

### Assessing Word Relevancy Via Term Frequency-Inverse Document Frequency

The scikit-learn library implements another transformer, the TfidfTransformer class, which takes the raw term frequencies from the CountVectorizer class as input and transforms them into tf-idfs:

In [8]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf = TfidfTransformer(use_idf=True, 
                         norm='l2',
                         smooth_idf=True)
np.set_printoptions(precision=2)
print(tfidf.fit_transform(count.fit_transform(docs)).toarray())

[[0.   0.43 0.   0.56 0.56 0.   0.43 0.   0.  ]
 [0.   0.43 0.   0.   0.   0.56 0.43 0.   0.56]
 [0.5  0.45 0.5  0.19 0.19 0.19 0.3  0.25 0.19]]


Scikit-learn's calculation for tf-idf is different from the default equation, the idf has a +1 on both the numerator and the denominator and the tf-idf multiplies by idf + 1 because we set the smooth_idf parameter to True, which is helpful for assigning zero weights to terms that occur in all documents.

While it is also more typical to normalize the raw term frequencies before calculating the tf-idfs, the TfidfTransformer class normalizes the tf-idfs directly. By default (norm = 'l2'), L2-normalization is applied, which returns a vector of length 1 by dividing an unnormalized feature vector by its L2-norm.

### Cleaning Text Data

Before we build our bag-of-words model, it is important to clean the text data by stripping it of all unwanted characters. To illustrate why this is important, let's display the last 50 characters from the first document in the reshuffled movie review dataset:

In [9]:
df.loc[0, 'review'][-50:]

'is seven.<br /><br />Title (Brazil): Not Available'

As we can see, the text contains HTML markup as well as punctuation and other non-letter characters. While HTML markup does not contain many useful semantics, punctuation marks can represent useful, additional information in certain NLP contexts. However, for simplicity, we will now remove all punctuation marks except for emoticon characters, such as :), since those are certainly useful for sentiment analysis. To accomplish this task, we will use the regex library re as shown here:

In [10]:
import re

def preprocessor(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', '')
    return text

Via the first regex, <[^>]*>, in the preceding code section, we tried to remove all of the HTML markup form the movie reviews. Although many people generally advise against the use of regex to parse HTML, this regex should be sufficient to clean this particular dataset. Since we are only interested in removing HTML markup and do not plan to use the HTML markup further, using regex to do the job should be acceptable. However, if you prefer to use sophisticated tools for removing HTML markup from text, you can take a look at Python's HTML parser module. After we removed the HTML markup, we used a slightly more complex regex to find emoticons, which we temporarily stored as emoticons. Next, we removed all non-word characters from the text via the regex [\W]+ and converted the text into lowercase characters.

Although the addition of the emoticon characters to the end of the cleaned document strings may not look like the most elegant approach, we must note that the order of the words doesn't matter in our bag-of-words model if our vocabulary consists of only one-word tokens. But before we talk more about the splitting of documents into individual terms, words, or tokens, let's confirm our preprocessor works correctly:

In [11]:
print(preprocessor(df.loc[0, 'review'][-50:]))
print(preprocessor("</a>This :) is :( a test :-)!"))

is seven title brazil not available
this is a test :) :( :)


Since we will make use of the cleaned text data over and over again during the next sections, let's apply our preprocessor to all the movie reviews in our DataFrame:

In [12]:
df['review'] = df['review'].apply(preprocessor)

### Processing Documents into Tokens

After successfully preparing the movie review dataset, we now need to think about how to split the text corpora into individual elements. One way to tokenize documents is to split them into individual words by splitting the cleaned documents at their whitespace characters:

In [13]:
def tokenizer(text):
    return text.split()

tokenizer('runners like running and thus they run')

['runners', 'like', 'running', 'and', 'thus', 'they', 'run']

The following code uses the Porter stemming algorithm and modifies our tokenizer function to reduce words to their root forms:

In [14]:
from nltk.stem.porter import PorterStemmer
porter = PorterStemmer()

def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

tokenizer_porter('runners like running and thus they run')

['runner', 'like', 'run', 'and', 'thu', 'they', 'run']

To remove stop words from the movie reviews, we will use the set of 127 English stop words that is available from the NLTK library, which can be obtained by calling the nltk.download function:

In [15]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Kaan
[nltk_data]     Oram\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
from nltk.corpus import stopwords
stop = stopwords.words('english')
[w for w in tokenizer_porter('a runner likes running and runs a lot') if w not in stop]

['runner', 'like', 'run', 'run', 'lot']

### Training a Logistic Regression Model for Document Classification

In this section, we will train a logistic regresion model to classify the movie reviews into positive and negative reviews based on the bag-of-words model. First, we will divide the DataFrame of cleaned text documents into 25,000 documents for training and 25,000 documents for testing:

In [17]:
X_train = df.loc[:25000, 'review'].values
y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
y_test = df.loc[25000:, 'sentiment'].values

Next, we will use a GridSearchCV object to find the optimal set of parameters for our logistic regression model using 5-fold stratified cross-validation:

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None,
                        token_pattern=None)

small_param_grid = [
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [None],
        'vect__tokenizer': [tokenizer, tokenizer_porter],
        'clf__l1_ratio': [0],
        'clf__C': [1.0, 10.0]
    },
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [stop, None],
        'vect__tokenizer': [tokenizer],
        'vect__use_idf': [False],
        'vect__norm': [None],
        'clf__l1_ratio': [0],
        'clf__C': [1.0, 10.0]
    }
]
lr_tfidf = Pipeline([
    ('vect', tfidf),
    ('clf', LogisticRegression(solver='liblinear'))
])
gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid,
                           scoring='accuracy',
                           cv=5,
                           verbose=2,
                           n_jobs=-1)
gs_lr_tfidf.fit(X_train, y_train)
print(f'Best parameter set: {gs_lr_tfidf.best_params_}')

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameter set: {'clf__C': 10.0, 'clf__l1_ratio': 0, 'vect__ngram_range': (1, 1), 'vect__stop_words': None, 'vect__tokenizer': <function tokenizer at 0x0000024DA64E2E80>}


As we can see, the best grid search results came from using the regular tokenizer without Porter stemming, no stop word library, and tf-idfs in combination with a logistic regression classifier that uses L2-regularization with the regularization strength C of 10. 

Using the best model from this grid search, let's print the average 5-fold cross validation accuracy scores on the training dataset and the classification accuracy on the test dataset:

In [19]:
print(f'CV Accuracy: {gs_lr_tfidf.best_score_:.3f}')
clf = gs_lr_tfidf.best_estimator_
print(f'Test Accuracy: {clf.score(X_test, y_test):.3f}')

CV Accuracy: 0.897
Test Accuracy: 0.899


The results reveal that our machine learning model can predict whether a movie review is positive or negative with 90% accuracy.

### Working with Bigger Data - Online Algorithms and Out-of-Core Learning

It could be computationally quite expensive to construct feature vectors for the 50,000-movie review dataset during a grid search. In many real-world applications, it is not uncommon to work with even larger datasets that can exceed our computer's memory.

We will now apply a technique called out-of-core learning, which allows us to work with such large datasets by fitting the classifier incrementally on smaller batches of a dataset.

In this section, we will make use of the partial_fit function of SGDClassifier in scikit-learn to stream the document directly from our local drive and train a logistic regression model using small mini-batches of documents. First, we will define a tokenizer function that cleans the unprocessed text data from the movie_data.csv file that we constructed at the beginning of this chapter and separates it into word tokens while removing stop words:

In [20]:
import numpy as np
import re
from nltk.corpus import stopwords
stop = stopwords.words('english')

def tokenizer(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', '')
    tokenized = [w for w in text.split() if w not in stop]
    return tokenized

Next, we will define a generator function, stream_docs, that reads in and returns one document at a time:

In [21]:
def stream_docs(path):
    with open(path, 'r', encoding='utf-8') as csv:
        next(csv) # skip header
        for line in csv:
            text, label = line[:-3], int(line[-2])
            yield text, label

To verify that our stream_docs function works correctly, let's read in the first document from the movie_data.csv file, which should return a tuple consisting of the review text as well as the corresponding class label:

In [22]:
next(stream_docs(path='movie_data.csv'))

('"In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70\'s, they discover the criminal and a net of power and money to cover the murder.<br /><br />""Murder in Greenwich"" is a good TV movie, with the true story of a murder of a fifteen years old girl that was committed by a wealthy teenager whose mother was a Kennedy. The powerful and rich f

We will now define a function, get_minibatch, that will take a document stream from the stream_docs function and return a particular number of documents specified by the size parameter:

In [23]:
def get_minibatch(doc_stream, size):
    docs, y = [], []
    try:
        for _ in range(size):
            text, label = next(doc_stream)
            docs.append(text)
            y.append(label)
    except StopIteration:
        return None, None
    return docs, y

Unfortunately, we can't use CountVectorizer for out-of-core learning since it requires holding the complete vocabulary in memory. Also, TfidfVectorizer needs to keep all the feature vectors of the training dataset in memory to calculate the inverse document frequencies. However, another useful vectorizer for text processing implemented in scikit-learn is HashingVectorizer. It is data independent and makes use of the hashing trick via the 32-bit MurmurHash3 function:

In [24]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
vect = HashingVectorizer(decode_error='ignore',
                           n_features=2**21,
                           preprocessor=None,
                           tokenizer=tokenizer)
clf = SGDClassifier(loss='log_loss', random_state=1)
doc_stream = stream_docs(path='movie_data.csv')

Using the preceding code, we initialized HashingVectorizer with our tokenizers function and set the number of features to $2^{21}$. Furthermore, we reinitialized a logistic regression classifier by setting the loss parameter of SGDClassifier to 'log_loss'. By choosing a large number of features in HashingVectorizer, we reduce chances of hash collisions, but we also increase the number of coefficients in our logistic regression model.

Now, we can start out-of-core learning using the following code:

In [25]:
classes = np.array([0, 1])
for _ in range(45):
    X_train, y_train = get_minibatch(doc_stream, size=1000)
    if not X_train:
        break
    X_train = vect.transform(X_train)
    clf.partial_fit(X_train, y_train, classes=classes)


We iterated over 45 mini-batches of documents where each mini-bath consists of 1000 documents. Having completed the incremental learning process, we will use the last 5000 documents to evaluate the performance of our model:

In [26]:
X_test, y_test = get_minibatch(doc_stream, size=5000)
X_test = vect.transform(X_test)
print(f'Accuracy: {clf.score(X_test, y_test):.3f}')

Accuracy: 0.868


As we can see, the accuracy of the model is approximately 87 percent, slightly below the accuracy that we achieved in the previous section using the grid search for hyperparameter tuning. However, out-of-core learning is very memory efficient, and it took less than a minute to complete.

Finally, we can use the last 5000 documents to update the model:

In [27]:
clf = clf.partial_fit(X_test, y_test)

### Topic Modeling with Latent Dirichlet Allocation

We will use the LatentDirichletAllocation class implemented in scikit-learn to decompose  the movie review dataset and categorize it into different topics. In the following example, we will restrict analysis to 10 different topics.

First, we are going to load the dataset into a pandas DataFrame using the local movie_data.csv file of the movie reviews that we created at the beginning of this chapter:

In [28]:
import pandas as pd
df = pd.read_csv('movie_data.csv', encoding='utf-8')

Next, we are going to use the CountVectorizer to create the bag-of-words matrix as input to LDA:

In [29]:
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer(stop_words='english',
                        max_df=.1,
                        max_features=5000)
X = count.fit_transform(df['review'].values)

Note that we set the maximum document frequency of words to be considered to 10 percent (max_df = 0.1) to exclude words that occur too frequently across documents. The rationale behind the removal of frequently occurring words is that these might be the common words appearing across all documents that are, therefore, less likely to be associated with a specific topic category of a given document. Also, we limited the number of words to be considered to the most frequently occurring 5000 words to limit the dimensionality of this dataset to improve the inference performed by LDA

The following code example demonstrates how to fit a LatentDirichletAllocation estimator to the bag-of-words matrix and infer the 10 different topics from the documents:

In [31]:
from sklearn.decomposition import LatentDirichletAllocation
lda = LatentDirichletAllocation(n_components=10,
                                random_state=123,
                                learning_method='batch')
X_topics = lda.fit_transform(X)

By setting learning_method to 'batch', we let the lda estimator do its estimation based on all available training data in one iteration, which is slower than the alternative 'online' learning method, but can lead to more accurate results.

After fitting the LDA, we now have access to the components_ attribute of the lda instance, which stores a matrix containing the word importance for each of the 10 topics in increasing order:

In [32]:
lda.components_.shape

(10, 5000)

To analyze the results, let's print the five most important words for each of the 10 topics. Note that the word importance values are ranked in increasing order. Thus, to print the top five words, we need to sort the topic array in reverse order:

In [33]:
n_top_words = 5
feature_names = count.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    print(f'Topic {topic_idx + 1}:')
    print(" ".join([feature_names[i] 
                    for i in topic.argsort()\
                    [:-n_top_words - 1:-1]]))

Topic 1:
horror worst script effects budget
Topic 2:
dvd watched video music guy
Topic 3:
war american series history documentary
Topic 4:
game killer murder thriller crime
Topic 5:
kids comedy episode series school
Topic 6:
family woman mother beautiful feel
Topic 7:
role performance comedy john plays
Topic 8:
action horror john effects dr
Topic 9:
book version original read music
Topic 10:
action wife father police james


To confirm that these categories make sense based on the reviews, let's plot three movies from the horror movie category:

In [39]:
horror = X_topics[:, 0].argsort()[::-1]
for iter_idx, movie_idx in enumerate(horror[:3]):
    print(f'\nHorror movie #{iter_idx + 1}:')
    print(f'{df["review"][movie_idx][:100]}...')


Horror movie #1:
Oh my God... where to begin? "Chupacabra Terror" is one of the worst B-Horror movies ever made. This...

Horror movie #2:
Carnosaur 3 is bad... awfully bad. Bad to the point where it is funny. How matter how much I try to ...

Horror movie #3:
Jack Frost 2, is probably the most cheesiest movie I have ever seen in my life. The complete title o...


Using the preceding example, we printed the first 100 characters from the top three horror movies. The reviews do look like horror movie reviews.